## Tâches d'inférence

Dans ce notebook, nous explorerons deux méthodes pour exécuter des tâches d'inférence sur les clusters de l'alliance :

* Sur Jupyter

* Avec une tâche batch

Dans les deux cas, le code d'entraînement sera fondamentalement le même. L'accent sera mis sur les différences et sur les points essentiels à connaître pour garantir un fonctionnement correct dans les deux cas.

### Qu'est-ce qu'un tâche d'inférence ?
L'inférence est le processus par lequel un modèle d'apprentissage automatique entraîné* tire des conclusions à partir de données inédites. Un modèle d'IA capable d'effectuer des inférences peut le faire sans exemples du résultat souhaité. Autrement dit, l'inférence est un modèle d'IA en action.

**Ce que ça fait:** Charge le fichier de modèle enregistré et prend de nouvelles entrées utilisateur non vues (comme une invite, une image ou une lecture de capteur) pour renvoyer une prédiction ou une réponse.

**System Requirements:** Les paramètres du modèle sont figés durant ce processus. Les requêtes utilisateur devant être rapides et réactives, l'accent est mis sur une faible latence et un coût avantageux plutôt que sur le calcul massivement parallèle.

**Outcome:** La prédiction est renvoyée à l'utilisateur ou à une application externe, fonctionnant comme un appel d'API classique.

### Inférence par IA vs. Entraînement
* L'entraînement est la première phase d'un modèle d'IA. Il peut s'agir d'un processus d'essais et d'erreurs, d'un processus consistant à présenter au modèle des exemples d'entrées et de sorties souhaitées, ou des deux.

* L'inférence est le processus qui suit l'entraînement d'une IA. Plus un modèle est entraîné et affiné, meilleures seront ses inférences, même si elles ne peuvent jamais être parfaitement exactes.

### VLLM
Nous nous concentrons principalement sur VLLM, un projet communautaire. Il support des algorithmes de décodage, la quantification, le parallélisme et les modèles de HuggingFace et d'autres sources.

VLLM est devenu l'un des moteurs d'inférence les plus utilisés pour le déploiement de grands modèles de langage. Ce framework vise principalement à optimiser l'utilisation du GPU pendant l'inférence. Son innovation majeure est PagedAttention, un système de gestion de la mémoire qui modifie la façon dont le cache clé-valeur est stocké et réutilisé.

### Comment installer VLLM
Pour voir la dernière version de VLLM:<br>
`avail_wheels vllm`

1. Charger les dépendances, charger les modules Python et OpenCV <br>
   <code>module load opencv/4.13 python/3.13</code>
3. Créer et démarrer un environnement virtuel temporaire<br>
   <code>virtualenv --no-download ~/ENV</code><br>
   <code>source ~/ENV/bin/activate</code>
5. Installez VLLM dans l'environnement virtuel ainsi que ses dépendances Python.<br>
   <code>pip install --no-index vllm==X.Y.Z</code>
7. Bloquer l'environnement et les exigences<br>
   <code>pip freeze > ~/vllm-requirements.txt</code>
9. Désactiver l'environnement<br>
    <code>deactivate</code>
11. Nettoyer et supprimer l'environnement virtuel<br>
    <code>rm -r ~/ENV</code>

### Soumission du tache
**Download model**<br>
Les modèles chargés pour l'inférence sur vLLM proviennent généralement du HuggingFace Hub.

Voici un exemple d'utilisation de l'outil en ligne de commande de HuggingFace pour télécharger un modèle. Veuillez noter que les modèles doivent être téléchargés sur un nœud de login afin d'éviter une utilisation inutile des ressources de calcul pendant le téléchargement. Par défaut, les modèles sont mis en cache dans le répertoire `$HOME/.cache/huggingface/hub`. 

```bash 
module load python/3.12
virtualenv --no-download temp_env && source temp_env/bin/activate
pip install --no-index huggingface_hub
hf download facebook/opt-125m
rm -r temp_env

**Remarque:** Étant donné que nous avons déjà téléchargé facebook/opt-125m dans le notebook précédent, nous copions simplement le chemin d'accès à l'endroit où il a été téléchargé et le collons dans le modèle ci-dessous.

### Tâches d'inférence dans Jupyter
Voici un exemple d'exécution de code Python effectuant une inférence sur un seul nœud GPU. Cet exemple suppose que vous avez préalablement téléchargé le modèle facebook/opt-125m (enregistré dans ~/scratch/facebook). Le nombre de GPU est contrôlé par le paramètre `tensor_parallel_size`.

In [ ]:
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel

# 1. Define input prompts
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

# 2. Initialize the LLM engine and load the opt-125m model
llm = LLM(model="/scratch/instructor3/facebook",tensor_parallel_size=1)

# 3. Define sampling parameters (temperature, top-p)
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)

# 4. Generate outputs
outputs = llm.generate(prompts,sampling_params)

# 5. Print results
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

# 5. Destroy the model and release the GPU memory
destroy_model_parallel()
llm.llm_engine.engine_core.shutdown()

### L'inference avec une tâche batch
Ici encore, nous exécutons le même code Python (voir ci-dessus) sur un seul nœud GPU. Mais cette fois-ci, nous n'avons pas besoin de Jupyter pour ce faire :

```bash
#!/bin/bash
#SBATCH --ntasks=1
#SBATCH --gpus-per-task=1
#SBATCH --cpus-per-task=1  
#SBATCH --mem=32000M       
#SBATCH --time=0-00:05
#SBATCH --output=%N-%j.out

module load python/3.13 opencv/4.13
virtualenv --no-download $SLURM_TMPDIR/env
source $SLURM_TMPDIR/env/bin/activate
pip install -r vllm-requiremnts.txt --no-index

python vllm-example.py

### Tâches d'inférence sur plusieurs nœuds de calcul
L'exemple suivant reprend celui du nœud unique présenté précédemment, mais répartit le modèle sur 4 GPU répartis sur 2 nœuds distincts, soit 2 GPU par nœud.

Actuellement, VLLM utilise Ray pour gérer la répartition des modèles sur plusieurs nœuds. L'exemple de code ci-dessous décrit les étapes nécessaires pour démarrer un cluster Ray multi-nœuds et y exécuter VLLM :

**vllm-multinode-example.sh**
```bash
#!/bin/bash
#SBATCH --nodes 2
#SBATCH --ntasks-per-node=1
#SBATCH --gpus-per-task=2
#SBATCH --cpus-per-task=6  
#SBATCH --mem=32000M       
#SBATCH --time=0-00:10
#SBATCH --output=%N-%j.out

## Create a virtualenv and install Ray on all nodes ##

module load gcc python/3.13 arrow/24 opencv/4.13

srun -N $SLURM_NNODES -n $SLURM_NNODES config_env.sh

export HEAD_NODE=$(hostname --ip-address) # store head node's address
export RAY_PORT=34567 # choose a port to start Ray on the head node 

## Set Huggingface libraries to OFFLINE mode ##

export HF_HUB_OFFLINE=1 
export TRANSFORMERS_OFFLINE=1

source $SLURM_TMPDIR/ENV/bin/activate

## Start Ray cluster Head Node ##
ray start --head --node-ip-address=$HEAD_NODE --port=$RAY_PORT --num-cpus=$SLURM_CPUS_PER_TASK --num-gpus=2 --block &
sleep 10

## Launch worker nodes on all the other nodes allocated by the job ##
srun launch_ray.sh &
ray_cluster_pid=$!
sleep 10

VLLM_HOST_IP=`hostname --ip-address` python vllm_example.py

## Shut down Ray worker nodes after the Python script exits ##
kill $ray_cluster_pid

**config_env.sh:**
```bash
#!/bin/bash

module load python/3.13 opencv/4.13 arrow/24
virtualenv --no-download $SLURM_TMPDIR/ENV
source $SLURM_TMPDIR/ENV/bin/activate
pip install --upgrade pip --no-index
pip install ray -r vllm-requirements.txt --no-index
deactivate

**launch_ray.sh:**
```bash
#!/bin/bash

if [[ "$SLURM_PROCID" -eq "0" ]]; then
        echo "Ray head node already started..."
        sleep 10

else
        export VLLM_HOST_IP=`hostname --ip-address`
        ray start --address "${HEAD_NODE}:${RAY_PORT}" --num-cpus="${SLURM_CPUS_PER_TASK}" --num-gpus=2 --block
        sleep 5
        echo "ray worker started!"
fi

**vllm_example.py:**
```python
from vllm import LLM

prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

# Set "tensor_parallel_size" to the TOTAL number of GPUs on all nodes.

llm = LLM(model="facebook/opt-125m",tensor_parallel_size=4)

outputs = llm.generate(prompts)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

### Alternatives à VLLM
* SGLang.
* TensorRT-LLM.
* Hugging Face Text Generation Inference (TGI)
* OpenLLM.
* DeepSpeed.
* Ollama.
* MLC LLM.
* llama. cpp.